In [ ]:
# 0. Imports
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A


from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# 1. Unzip Dataset + Auto Path Fix
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("Extraction completed.")
else:
    print("Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("Using dataset folder:", extract_path)
print("First 5 contents:", os.listdir(extract_path)[:5])

# 2. Load Data Function
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# 3. Load Dataset
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("Images:", images.shape)
print("Masks:", masks.shape)

# 4. Loss Functions
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# 5. Convolution Blur + Hybrid Attention (CBA)
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=3, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=3,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# 6. Transformer Encoder Components
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# 7. TransUNet with Hybrid CBA
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# 8. Metrics
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# 9. K-Fold Training
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\nFold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# 10. Final Results
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\nFinal Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")